# 04 - Decoding & evaluation (Colab driver)

Thin driver: load a trained checkpoint -> **decode** the test set (greedy or beam) -> report **BLEU + chrF++ + TER**.

**Before running:**
- GPU runtime (Runtime -> Change runtime type -> GPU).
- `01_data` must have produced the cleaned eval files + tokenizer on Drive under `DATA_ROOT` (`test.clean.hi/.en`, `dev.clean.hi/.en`, `spm_unigram_16000.model`).
- Point `CKPT_PATH` at a trained `best.pt`. The smoke run wrote to local `/content` (wiped between sessions), so for a real eval save your run's `checkpoints/` to Drive and set the path.

*Decode mode is set by `DecodeConfig.mode` (`"greedy"` or `"beam"`); MBR / ensemble and the manual-eval dump come later.*

## 1. Repo + install

In [1]:
import os, sys

REPO_URL = "https://github.com/Rishi-Jain-27/hindi-translator.git"
REPO_DIR = "/content/hindi-translator"

!test -d {REPO_DIR} && git -C {REPO_DIR} pull || git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.abspath("src"))
print("cwd:", os.getcwd())

Cloning into '/content/hindi-translator'...
remote: Enumerating objects: 457, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 457 (delta 7), reused 16 (delta 6), pack-reused 434 (from 1)
Receiving objects: 100% (457/457), 13.75 MiB | 26.07 MiB/s, done.
Resolving deltas: 100% (229/229), done.
cwd: /content/hindi-translator


In [2]:
!pip install -e .

Obtaining file:///content/hindi-translator
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 107.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 16.2 MB/s eta 0:00:00
  Building editable for nmt (pyproject.toml) ... done
  Created wheel for nmt: filename=nmt-0.1.0-0.editable-py3-none-any.whl size=1279 sha256=011f9b96097820ea0814597dc6d2fc9ebec2917c1928b7d61811aad6243815df
  Stored in directory: /tmp/pip-ephem-wheel-cache-5k62bm7x/wheels/bd/c2/ef/ae818ff943d77ea8d63ef07aea61a1b82808716362dc169d4c
Successfully built nmt


In [3]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

device: cuda NVIDIA A100-SXM4-80GB


## 2. Config + Drive

`ModelConfig` must match what was trained. `DecodeConfig` drives greedy decoding (batch size + max generated length).

In [5]:
from nmt.config import DataConfig, ModelConfig, DecodeConfig
from google.colab import drive

# same Drive folder 01_data wrote the corpus + tokenizer to
drive.mount("/content/drive")
DATA_ROOT = "/content/drive/MyDrive/hindi-translator/data"

data_cfg = DataConfig(raw_dir=f"{DATA_ROOT}/raw", cache_dir=f"{DATA_ROOT}/cache")
model_cfg = ModelConfig()                        # MUST match the trained model (base: 512 / 8 heads / 6+6)
decode_cfg = DecodeConfig(
    mode="beam",                                 # "greedy" | "beam"  (mbr/ensemble not wired yet)
    beam_size=5,                                 # beam width (mode="beam"); beam_size=1 == greedy
    length_penalty=0.6,                          # GNMT length normalization (mode="beam")
    batch_size=64,                               # greedy: sentences per batch. beam runs 1 sentence at a time
    max_decode_len=128,                          # hard cap on generated length; raise if outputs look cut off
)

# point at a TRAINED best.pt. the smoke run saved to local /content (wiped on reset);
# for a real eval, save your run's checkpoints/ to Drive and set this path.
CKPT_PATH = "/content/drive/MyDrive/hindi-translator/checkpoints/best.pt"
SPLIT = "test"                                   # "test" (~2507 pairs) or "dev" (~520, faster smoke)
print("ckpt:", CKPT_PATH, "| mode:", decode_cfg.mode, "| split:", SPLIT)

Mounted at /content/drive
ckpt: /content/drive/MyDrive/hindi-translator/checkpoints/best.pt | mode: beam | split: test


## 3. Tokenizer + model

In [6]:
from nmt.data.tokenizer import Tokenizer
from nmt.model.transformer import build_model

cache = data_cfg.cache_dir
tok = Tokenizer.load(os.path.join(cache, f"spm_{data_cfg.tokenizer_model}_{data_cfg.vocab_size}.model"))
print("tokenizer vocab:", tok.vocab_size)

model = build_model(model_cfg, tok).to(device)   # copies vocab_size + special ids from the tokenizer
print("params:", f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M")

tokenizer vocab: 16000
params: 52.3M


## 4. Load trained weights (EMA)

Load `best.pt`, then overlay the **EMA shadow** weights - those are what the training loop measured dev loss on (the raw weights jitter). Falls back to raw weights if the checkpoint has no EMA.

In [7]:
from nmt.train.ema import EMA

ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model"])             # raw weights, incl. buffers (e.g. positional encodings)

if ckpt.get("ema"):
    ema = EMA(model)
    ema.load_state_dict(ckpt["ema"])
    ema.copy_to(model)                           # overlay the EMA params onto the model
    print("applied EMA weights")
else:
    print("no EMA in checkpoint -> using raw weights")

model.eval()
print("checkpoint step:", ckpt["step"], "| best dev nll:", ckpt.get("best"))

applied EMA weights
checkpoint step: 16000 | best dev nll: 1.9370505617336886


## 5. Read the eval set

Two line-aligned cleaned files -> source + reference lists. `read_lines` drops the trailing-newline empty and raises if the two files disagree in length.

In [8]:
from nmt.eval.evaluate import read_lines

src_path = os.path.join(cache, f"{SPLIT}.clean.hi")
ref_path = os.path.join(cache, f"{SPLIT}.clean.en")
srcs, refs = read_lines(src_path, ref_path)
print(f"{SPLIT}: {len(srcs)} sentence pairs")

test: 2507 sentence pairs


## 6. Translate + score

Decode every source (mode set above) and score against the references. **Note:** beam search runs **one sentence at a time** (greedy decodes a whole batch at once), so beam over the full test set is noticeably slower — try `SPLIT="dev"` first for a quick read, and lower `beam_size` if it drags. Greedy with this checkpoint scored ~19.3 BLEU; beam should add a point or two.

In [9]:
from nmt.eval.evaluate import evaluate

results = evaluate(model, tok, srcs, refs, decode_cfg)
results

bleu: 20.067967195247892
chrF: 49.07602685309696
TER: 68.27871009534606


{'bleu': 20.067967195247892,
 'chrF': 49.07602685309696,
 'TER': 68.27871009534606}

## 7. Eyeball a few translations

Source / model output / reference for the first few sentences - a quick sanity check beyond the aggregate numbers.

In [10]:
from nmt.decode.translate import translate

sample_src = srcs[:10]
sample_hyp = translate(model, sample_src, tok, decode_cfg)
for s, h, r in zip(sample_src, sample_hyp, refs[:10]):
    print("SRC:", s)
    print("HYP:", h)
    print("REF:", r)
    print("-" * 70)

SRC: आपकी कार में ब्लैक बॉक्स?
HYP: Black box in your car?
REF: A black box in your car?
----------------------------------------------------------------------
SRC: जबकि अमेरिका के सड़क योजनाकार, ध्वस्त होते हुए हाईवे सिस्टम को सुधारने के लिए धन की कमी से जूझ रहे हैं, वहीं बहुत-से लोग इसका समाधान छोटे से ब्लैक बॉक्स में देख रहे हैं, जो आपकी कार के डैशबोर्ड पर सफ़ाई से फिट हो जाता है।
HYP: While the US road planners are struggling with money scarcity to improve the highway system, many people are looking at the solution from the small black box that fits into the Dashboard of your car.
REF: As America's road planners struggle to find the cash to mend a crumbling highway system, many are beginning to see a solution in a little black box that fits neatly by the dashboard of your car.
----------------------------------------------------------------------
SRC: यह डिवाइस, जो मोटर-चालक द्वारा वाहन चलाए गए प्रत्येक मील को ट्रैक करती है तथा उस सूचना को अधिकारियों को संचारित करती है, आजकल अमेरिक